# XGBoost: Train and Compare

The random forest votes over 200 independent trees. **Boosting** builds trees **sequentially, each one fitted to the mistakes left by the trees before it.** Each correction is scaled down by the `learning_rate`, so no single tree moves the forecast too far.

| | Random forest | XGBoost (boosting) |
|---|---|---|
| Trees are built | Independently, all at once | One after another |
| Each tree | Deep, learns everything it can | Shallow (3–7 questions), fixes the remaining mistakes |
| Final forecast | Vote of all trees | Sum of all small corrections |
| More trees | Never overfits, only costs time | **Eventually overfits**: it starts correcting noise |

Because of the last row, XGBoost needs **early stopping**: add trees while the score on unseen days improves, stop when it does not.

**The bar** (set in `dt_train.ipynb`, section 9): a model only counts as better if it beats persistence **on average over the five years 2019–2023 and in most of them**. The random forest is the first model to pass it (`rf_train.ipynb`); XGBoost has to match or beat the forest.

Decisions so far: [`docs/model_decisions.md`](../docs/model_decisions.md). Run `lr_train_baseline.ipynb`, `dt_train.ipynb` and `rf_train.ipynb` first: this notebook loads their saved models to compare with them.

## 1. Setup

XGBoost requires the categories as integers 0–5. `CategoryXGB` in `utility.py` wraps it, encoding the names before training and decoding them on prediction, so XGBoost fits in a `Pipeline` and in `rolling_validation` like every other model.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

import sys
sys.path.insert(0, '../scripts')                 # utility.py lives in scripts/
from utility import (AQI_CATEGORIES, prepare_dataset, split_train_val_test, add_lr_features,
                     forecast_scorecard, class_scores_table, rolling_validation, rolling_summary,
                     CategoryXGB)

DATA_PATH = '../data/processed/aqi_weather_2015_2025.csv'
MODEL_DIR = Path('models')
TARGET = 'tomorrow_status'
TREND = ['aqi_roll7_mean', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7']
pd.set_option('display.max_columns', 30)

## 2. The same days as every other model

In [ ]:
df = prepare_dataset(DATA_PATH)
before = len(df)
df = df[df[TREND].notna().all(axis=1)]                   # the same 195 rows every model drops
df_train, df_val, df_test = split_train_val_test(df)

print(f'Dropped {before - len(df)} rows with empty trend features')
print('train:', len(df_train), '| validation:', len(df_val), '| test:', len(df_test))

## 3. Features: the same raw columns as the trees

The decision tree's and the forest's raw features, no transforms.

Early stopping needs the 2023 days already vectorised during training, so `DictVectorizer` is fitted once here, outside a pipeline. The final model (section 8) puts it back into a pipeline, so it still accepts the dictionaries the web service sends.

In [ ]:
TREE_NUMERIC = (['aqi_today'] + TREND +
                ['temp_mean', 'temp_max', 'temp_min', 'temp_range', 'humidity_mean', 'dew_point_mean',
                 'precip_sum', 'pressure_mean', 'cloud_cover_mean', 'wind_speed_mean', 'wind_speed_max',
                 'wind_u_mean', 'wind_v_mean', 'month'])
TREE_BINARY = [c for c in df.columns if c.startswith('has_')] + ['aqi_imputed']
TREE_CATEGORICAL = ['city']
FEATURES = TREE_NUMERIC + TREE_BINARY + TREE_CATEGORICAL

def to_records(part):
    return part[FEATURES].to_dict(orient='records')

X_train, X_val = to_records(df_train), to_records(df_val)
y_train = df_train[TARGET].astype(str).to_numpy()
y_val = df_val[TARGET].astype(str).to_numpy()

dv = DictVectorizer(sparse=False).fit(X_train)
X_train_num, X_val_num = dv.transform(X_train), dv.transform(X_val)   # numbers only, for early stopping

assert df_train[FEATURES].notna().all().all() and df_val[FEATURES].notna().all().all(), 'missing values in the features'
print(f'{len(FEATURES)} features -> {X_train_num.shape[1]} columns after one-hot encoding the city')

## 4. The settings of XGBoost

| Setting | Meaning | Effect |
|---|---|---|
| `n_estimators` | Number of boosting rounds (trees) | Too few: underfits. Too many: fits noise. Chosen by early stopping |
| `learning_rate` | Share of each correction that is kept | Smaller → more rounds needed, usually slightly better. Fixed at 0.05 here |
| `max_depth` | Maximum tree depth | Boosting uses **shallow** trees (3–7): each fixes only part of the mistakes |
| `min_child_weight` | Roughly, minimum evidence per leaf | Boosting counterpart of `min_samples_leaf`: higher → fewer leaves fitted to odd days |
| `subsample`, `colsample_bytree` | Share of days and of features each tree sees (0.8 = 80%) | The forest's two ideas, used to decorrelate the trees |
| `balanced` | Weight every day by its category (wrapper option) | Equivalent to `class_weight='balanced'` |

In [ ]:
LEARNING_RATE = 0.05

def make_xgb(n_estimators, max_depth, min_child_weight, balanced=False):
    return Pipeline([
        ('dv', DictVectorizer(sparse=False)),             # city -> one-hot; numbers pass through
        ('xgb', CategoryXGB(n_estimators=n_estimators, learning_rate=LEARNING_RATE, max_depth=max_depth,
                            min_child_weight=min_child_weight, subsample=0.8, colsample_bytree=0.8,
                            balanced=balanced)),
    ])

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=AQI_CATEGORIES, average='macro')

## 5. Learning curve without early stopping

One model (trees of depth 5) trained for 1,000 rounds **without** stopping. After every round XGBoost records the **log loss** (its training objective; lower is better) on the training years and on 2023.

In [ ]:
curve_model = CategoryXGB(n_estimators=1000, learning_rate=LEARNING_RATE, max_depth=5, min_child_weight=1,
                          subsample=0.8, colsample_bytree=0.8)
curve_model.fit(X_train_num, y_train, eval_set=[(X_train_num, y_train), (X_val_num, y_val)])

history = curve_model.booster_.evals_result()
loss = pd.DataFrame({'training years': history['validation_0']['mlogloss'],
                     '2023': history['validation_1']['mlogloss']})
loss.index = loss.index + 1                               # round 1, 2, ...
best_round = loss['2023'].idxmin()

ROUNDS = [10, 50, 100, 200, 400, 600, 800, 1000]          # macro F1 if we had stopped after n rounds
f1_by_round = pd.DataFrame({
    n: {'training years': macro_f1(y_train, curve_model.classes_[curve_model.booster_.predict_proba(X_train_num, iteration_range=(0, n)).argmax(axis=1)]),
        '2023': macro_f1(y_val, curve_model.classes_[curve_model.booster_.predict_proba(X_val_num, iteration_range=(0, n)).argmax(axis=1)])}
    for n in ROUNDS})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
loss.plot(ax=axes[0])
axes[0].axvline(best_round, color='grey', linestyle=':', label=f'lowest 2023 loss (round {best_round})')
axes[0].set_xlabel('round (number of trees)')
axes[0].set_ylabel('log loss (lower is better)')
axes[0].legend()
f1_by_round.T.plot(ax=axes[1], marker='o')
axes[1].axhline(0.651, color='grey', linestyle='--', linewidth=1, label='persistence (0.651)')
axes[1].set_xlabel('round (number of trees)')
axes[1].set_ylabel('macro F1')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Lowest 2023 log loss at round {best_round}')
f1_by_round.round(3)

**Findings:**
- **The training score keeps climbing** (0.537 after 10 rounds, 0.922 after 1,000): with enough rounds XGBoost fits the training years ever more closely, like the unconstrained tree.
- **The 2023 score rises, peaks, then falls:** 0.668 after 100 rounds, best around **400 rounds** (0.674; the lowest 2023 log loss is at round 407), down to 0.659 after 1,000. The later trees fit noise in the training years.
- **Unlike the forest, too many rounds overfit**, so the number of rounds must be chosen. Early stopping does this in the next section.

## 6. Tuning on 2023, with early stopping

3 depths × 2 `min_child_weight` × 2 weightings = 12 models. Each may grow up to 1,500 trees, but **stops when the 2023 log loss has not improved for 50 rounds** and keeps the best round (`best_n_trees`). About 3 minutes.

Early stopping uses 2023 to pick the number of trees, as 2023 picked the tree depth before. The five-year check in section 7 keeps that number fixed rather than re-tuning it on each year.

In [ ]:
DEPTHS = [3, 5, 7]
CHILD_WEIGHTS = [1, 10]
WEIGHTS = {'no class weights': False, 'balanced': True}

rows = []
for weight_name, balanced in WEIGHTS.items():
    for depth in DEPTHS:
        for child_weight in CHILD_WEIGHTS:
            m = CategoryXGB(n_estimators=1500, learning_rate=LEARNING_RATE, max_depth=depth,
                            min_child_weight=child_weight, subsample=0.8, colsample_bytree=0.8, balanced=balanced)
            m.fit(X_train_num, y_train, eval_set=[(X_val_num, y_val)], early_stopping_rounds=50)
            pred_va = m.predict(X_val_num)
            rows.append({
                'weights': weight_name, 'max_depth': depth, 'min_child_weight': child_weight,
                'best_n_trees': m.best_n_trees_,
                'train_macro_f1': macro_f1(y_train, m.predict(X_train_num)),
                'val_macro_f1': macro_f1(y_val, pred_va),
                'said_severe': (pred_va == 'Severe').sum(),
            })

tuning = pd.DataFrame(rows)
tuning['gap'] = tuning['train_macro_f1'] - tuning['val_macro_f1']
print(len(tuning), 'models trained | real Severe days in 2023:', (y_val == 'Severe').sum())
tuning.sort_values('val_macro_f1', ascending=False).round(3)

**Findings:**
- **Without class weights, every setting beats persistence on 2023** (0.665 to 0.678).
- **Balanced weights hurt again on 2023** (0.649 to 0.663). They predict "Severe" 66 to 83 times for 51 real Severe days, the same over-forecasting as logistic regression, and need far more rounds (584 up to the 1,500 limit) before the 2023 loss stops improving.
- **Deeper trees need fewer rounds:** about 630 at depth 3, about 230 at depth 7, since each deeper tree corrects more at once.
- **The top three are within 0.010,** about the size of year-to-year noise, so five years choose.

## 7. Let five years choose

Shortlist: the **3 best settings of each weighting** on 2023, each with its own early-stopped number of trees, run through rolling validation (2019 to 2023) with the saved **random forest** and **decision tree** as reference. Same selection rule as the forest: among the XGBoost models that pass the bar and are within 0.002 of the best five-year average, prefer more years ahead of persistence, then the smaller training gap. About 5 minutes.

In [ ]:
with open(MODEL_DIR / 'rf_best.bin', 'rb') as f:        # saved by rf_train.ipynb, section 10
    rf_bundle = pickle.load(f)
with open(MODEL_DIR / 'dt_best.bin', 'rb') as f:        # saved by dt_train.ipynb, section 10
    dt_bundle = pickle.load(f)
with open(MODEL_DIR / 'lr_best.bin', 'rb') as f:        # saved by lr_train_baseline.ipynb
    lr_bundle = pickle.load(f)

SHORTLIST = (tuning.sort_values('val_macro_f1', ascending=False)
                   .groupby('weights', sort=False).head(3)
                   .copy())
SHORTLIST['name'] = SHORTLIST.apply(
    lambda r: f"xgboost: {r['weights']}, depth {r['max_depth']}, min_child_weight {r['min_child_weight']}, "
              f"{r['best_n_trees']} trees", axis=1)

def xgb_maker(row):                                      # a function that returns a fresh, unfitted model
    return lambda: make_xgb(int(row['best_n_trees']), int(row['max_depth']), int(row['min_child_weight']),
                            WEIGHTS[row['weights']])

def make_rf():                                           # the chosen random forest, same settings as rf_best.bin
    return Pipeline([('dv', DictVectorizer(sparse=False)),
                     ('forest', RandomForestClassifier(**rf_bundle['params'], n_jobs=-1, random_state=1))])

def tree_xy(train, val):
    return to_records(train), to_records(val)

CANDIDATES = {row['name']: xgb_maker(row) for _, row in SHORTLIST.iterrows()}
CANDIDATES['random forest (rf_best)'] = make_rf

df_dev = pd.concat([df_train, df_val])                   # 2015-2023 only; the test years stay locked
rolling = pd.concat({name: rolling_validation(df_dev, make, tree_xy) for name, make in CANDIDATES.items()},
                    names=['model']).reset_index(level=0)
SHORTLIST[['name', 'val_macro_f1', 'said_severe']].round(3)

**Macro F1 per year** (persistence first, for reference):

In [ ]:
by_year = rolling.pivot_table(index='model', columns='val_year', values='macro_f1')
persistence_row = rolling.groupby('val_year')['persistence_macro_f1'].first().rename('persistence')
by_year = pd.concat([by_year, persistence_row.to_frame().T]).reindex(['persistence'] + list(CANDIDATES))
by_year.round(3)

**Summary over the five years:**

In [ ]:
five_years = rolling_summary(rolling)
five_years.round(3)

In [ ]:
passing = five_years[five_years.index.str.startswith('xgboost') & five_years['passes_bar']]
assert len(passing), 'no XGBoost model passes the bar'
close = passing[passing['mean_macro_f1'] >= passing['mean_macro_f1'].max() - 0.002]    # within 0.002 = a tie
chosen_name = close.sort_values(['years_won', 'mean_train_gap'], ascending=[False, True]).index[0]
chosen = SHORTLIST.set_index('name').loc[chosen_name]
forest_mean = five_years.loc['random forest (rf_best)', 'mean_macro_f1']

print(f"Chosen: {chosen.name}")
print(f"  five-year mean {five_years.loc[chosen.name, 'mean_macro_f1']:.3f} "
      f"(random forest {forest_mean:.3f}, persistence {by_year.loc['persistence'].mean():.3f}), "
      f"ahead of persistence in {five_years.loc[chosen.name, 'years_won']} of {five_years.loc[chosen.name, 'years']} years")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for name in ['persistence', 'random forest (rf_best)', chosen.name]:
    style = dict(color='grey', linestyle='--') if name == 'persistence' else {}
    ax.plot(by_year.columns, by_year.loc[name], marker='o', label=name, **style)
ax.set_xticks(list(by_year.columns))
ax.set_xlabel('validation year (learned from all earlier years)')
ax.set_ylabel('macro F1')
ax.set_title('Rolling validation: XGBoost vs random forest vs persistence')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Findings:**
- **XGBoost passes the bar too.** All 6 shortlisted settings beat persistence on the five-year average (+0.011 to +0.022) and in at least 4 of the 5 years.
- **Chosen XGBoost** (no class weights, depth 3, `min_child_weight` 10, 628 trees): **0.655** against persistence's 0.634, **ahead in all 5 years**. The depth-7 setting averaged 0.001 higher (a tie) but lost 2022 and has a larger training gap, so the rule picked depth 3.
- **XGBoost does not beat the random forest.** The forest averages **0.664** and is ahead of the chosen XGBoost in all five years (in 2019 and 2021 only by 0.002 or less). The largest difference is again **2022** (0.639 against 0.621).
- **2022 is still the hardest year.** Unweighted XGBoost only ties or loses there (0.602 to 0.621 against persistence's 0.619); the balanced settings are only slightly ahead (+0.005). Only the forest is clearly ahead in 2022 (+0.020).
- **Balanced weights again look different over five years than on 2023.** On 2023 they lost clearly (0.663 against 0.678); over five years the best balanced setting is close (0.654 against 0.656) and steadier (spread 0.025). One year is not enough to judge a setting.

## 8. Compare on 2023: all five forecasts

The chosen XGBoost is retrained on 2015–2022 (inside a pipeline, with its fixed number of trees) and scored on 2023 with the same scorecard as every other model.

In [ ]:
booster = make_xgb(int(chosen['best_n_trees']), int(chosen['max_depth']), int(chosen['min_child_weight']),
                   WEIGHTS[chosen['weights']]).fit(X_train, y_train)
pred_xgb = booster.predict(X_val)

df_val_lr = add_lr_features(df_val, pd.Series(lr_bundle['city_pressure']))   # the LR needs its own transforms
lr_features = sum(lr_bundle['features'].values(), [])

PREDICTIONS = {
    'persistence': df_val['status_today'].astype(str).to_numpy(),
    'logistic regression': lr_bundle['model'].predict(df_val_lr[lr_features].to_dict(orient='records')),
    'decision tree': dt_bundle['model'].predict(X_val),
    'random forest': rf_bundle['model'].predict(X_val),
    'xgboost': pred_xgb,
}
scorecard = forecast_scorecard(y_val, PREDICTIONS, y_today=df_val['status_today'])
scorecard

**Precision, recall and F1 for every category:**

In [ ]:
class_scores_table(y_val, PREDICTIONS).round(2)

**Confusion matrix of XGBoost** (rows: what really happened, columns: what XGBoost forecast):

In [ ]:
pd.DataFrame(confusion_matrix(y_val, pred_xgb, labels=AQI_CATEGORIES),
             index=pd.Index(AQI_CATEGORIES, name='actual'),
             columns=pd.Index(AQI_CATEGORIES, name='predicted'))

**Findings:**

| 2023 | Persistence | Logistic regression | Decision tree | Random forest | XGBoost |
|---|---|---|---|---|---|
| Macro F1 | 0.651 | 0.647 | 0.658 | **0.679** | 0.668 |
| Accuracy | 0.710 | 0.729 | 0.723 | 0.734 | **0.735** |
| Severe caught / missed / false alarms | 27 / 24 / 25 | 20 / 31 / **11** | 25 / 26 / 19 | **31 / 20** / 35 | 24 / 27 / 18 |
| Dangerous recall / precision | 0.749 / 0.753 | 0.807 / 0.761 | 0.768 / 0.740 | **0.840** / 0.760 | 0.809 / **0.767** |
| Change days right | 0% | **35.3%** | 27.8% | 26.8% | 28.0% |
| Unchanged days right | **100%** | 88.3% | 90.4% | 92.4% | 92.1% |

- **XGBoost is second on 2023:** macro F1 0.668, ahead of persistence (+0.017), logistic regression and the tree, behind the forest (0.679); accuracy 73.5%, tied with the forest.
- **Conservative on Severe:** 18 false alarms, half the forest's 35, but only 24 of the 51 Severe days caught (forest 31): the trade-off between no weights (XGBoost) and balanced weights (forest).
- **Most precise Dangerous alert so far:** warns on 81% of Very Poor and Severe days at 77% precision; the forest warns on more (84%) at almost the same precision (76%).
- **Both trade-offs can be moved** by the planned **Severe threshold** step (how sure a model must be before it says "Severe"), so the five-year macro F1 is the fairer ranking, and there the forest leads.

## 9. Feature importance

XGBoost's feature importance (each feature's share of the total improvement) next to the forest's:

In [ ]:
importance = pd.DataFrame({
    'xgboost': pd.Series(booster['xgb'].booster_.feature_importances_, index=booster['dv'].feature_names_),
    'random forest': pd.Series(rf_bundle['model']['forest'].feature_importances_,
                               index=rf_bundle['model']['dv'].feature_names_),
})
importance.sort_values('xgboost', ascending=False).head(12).round(3)

**Findings:**
- **`aqi_today` again does the most work** (37%), followed by the **weekly average** (12%) and **rain** (`precip_sum`, 4%), which washes pollution out of the air.
- **The two columns are not measured the same way.** XGBoost reports by default the *average* improvement per split on a feature ("gain"), so a rarely used, high-impact feature scores high: `city=Kolkata` and `has_co` appear in the top 12 although they matter on few days. The forest reports the total improvement. Compare the order, not the exact numbers.

## 10. Save the model

In [ ]:
MODEL_PATH = MODEL_DIR / 'xgb_best.bin'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
bundle = {
    'model': booster,                                      # DictVectorizer + CategoryXGB (from utility.py)
    'params': {'n_estimators': int(chosen['best_n_trees']), 'learning_rate': LEARNING_RATE,
               'max_depth': int(chosen['max_depth']), 'min_child_weight': int(chosen['min_child_weight']),
               'subsample': 0.8, 'colsample_bytree': 0.8, 'balanced': WEIGHTS[chosen['weights']]},
    'features': {'numeric': TREE_NUMERIC, 'binary': TREE_BINARY, 'categorical': TREE_CATEGORICAL},
    'categories': AQI_CATEGORIES,
    'cities': sorted(df_train['city'].unique()),           # the service must reject any other city
    'validation': scorecard['xgboost'].to_dict(),
    'five_years': five_years.loc[chosen.name].to_dict(),
    'tuning': tuning.round(4).to_dict(orient='records'),
}
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(bundle, f)
with open(MODEL_PATH, 'rb') as f:
    assert (pickle.load(f)['model'].predict(X_val) == pred_xgb).all()
print(f'Saved and verified: {MODEL_PATH.resolve()} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)')

## Discussion

1. **XGBoost passes the bar.** Over 2019–2023 it averages **0.655** against persistence's **0.634**, ahead in **all five years**.
2. **It does not beat the random forest.** The forest averages **0.664** and is ahead of XGBoost in all five years; on 2023 it scores 0.679 against 0.668.
3. **Boosting overfits with too many rounds.** The 2023 score fell after about 400 rounds while the training score kept rising. Early stopping chose the number of rounds; the forest did not need it.
4. **Different Severe trade-offs:** XGBoost raises half as many false alarms (18 against 35) but catches fewer Severe days (24 against 31). The Severe threshold step can move either model along that trade-off.
5. **XGBoost is lighter:** a 4 MB model file against the forest's 114 MB.

**Where all models stand (rolling validation 2019–2023, macro F1):**

| Model | Five-year average | vs persistence | Years ahead | Passes the bar |
|---|---|---|---|---|
| Persistence (tomorrow = today) | 0.634 | — | — | — |
| Logistic regression | 0.613 | −0.021 | 1 of 5 | No |
| Decision tree | 0.631 | −0.003 | 3 of 5 | No |
| XGBoost | 0.655 | +0.021 | 5 of 5 | **Yes** |
| **Random forest** | **0.664** | **+0.030** | **5 of 5** | **Yes** |

**The random forest is the leading model.** Next steps:
1. **Severe threshold** on the forest's probability for Severe, to choose the balance between Severe days caught and false alarms.
2. **Final check:** retrain on 2015–2023 and score the locked test years 2024–2025 **once**.
3. `train.py` and `predict.py`, then the web service.